# PYROTECH VISION on Google Colab - live RTSP detection over a private VPN

**Before running:** on your PC on the camera LAN, run `scripts/colab/local_relay.py setup` and `start`
(see `docs/COLAB_DEPLOYMENT.md`). In Colab: *Runtime > Change runtime type > GPU* (optional - CPU works too).

Secrets (left sidebar, key icon) - give this notebook access to both:
* `TS_AUTHKEY` - Tailscale auth key, **ephemeral + single use** (Tailscale admin > Settings > Keys)
* `RELAY_PASSWORD` - printed by `local_relay.py setup`

Run the cells top to bottom. The last cell stops everything cleanly.

In [ ]:
# 1. Get / update the project
import os
REPO = "/content/pyrotech-vision-"
if not os.path.isdir(REPO):
    !git clone -q https://github.com/pyrotechhumandetection-lab/pyrotech-vision-.git {REPO}
else:
    !git -C {REPO} pull -q --ff-only
%cd {REPO}
!python3 scripts/colab/colab_launch.py check

In [ ]:
# 2. Relay settings (values printed by local_relay.py on your PC). Secrets come from Colab Secrets, never from this file.
import os, getpass
os.environ["RELAY_HOST"]  = "100.x.y.z"   # <- Tailscale address of your PC (local_relay.py show)
os.environ["RELAY_PORT"]  = "8554"
os.environ["RELAY_USER"]  = "aivision"
os.environ["RELAY_PATHS"] = "cam1"        # comma-separated relay paths
def secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return getpass.getpass(f"{name}: ")
os.environ["TS_AUTHKEY"] = secret("TS_AUTHKEY")
os.environ["RELAY_PASSWORD"] = secret("RELAY_PASSWORD")
print("settings loaded (secrets not shown)")

In [ ]:
# 3. Install: backend/.venv (CUDA PyTorch on a GPU runtime), Tailscale, socat - 5-10 min the first time
!python3 scripts/colab/colab_launch.py install

In [ ]:
# 4. Join your Tailscale network (userspace mode, node is removed again on 'stop')
!python3 scripts/colab/colab_launch.py vpn

In [ ]:
# 5. Bridge 127.0.0.1:18554 -> relay on your PC, then test: RTSP answer + one decoded frame
!python3 scripts/colab/colab_launch.py bridge
!python3 scripts/colab/colab_launch.py test-camera

In [ ]:
# 6. Start backend (GPU when available) + web UI, register the relay camera(s)
!python3 scripts/colab/colab_launch.py start
!python3 scripts/colab/colab_launch.py add-camera

In [ ]:
# 7. Open the web UI through Colab's authenticated proxy (only your Google account can open it)
from google.colab.output import serve_kernel_port_as_window
serve_kernel_port_as_window(5173, path="/")

In [ ]:
# 8. Optional: start Human Detection on the first relay camera from here (or use the UI)
import json, urllib.request, os
cid = "COLAB-" + os.environ["RELAY_PATHS"].split(",")[0].strip().upper()
req = urllib.request.Request("http://127.0.0.1:8000/api/detections/human/start", method="POST",
                             data=json.dumps({"cameraId": cid}).encode(), headers={"Content-Type": "application/json"})
print(urllib.request.urlopen(req, timeout=120).read()[:200])

In [ ]:
# 9. Status: services, GPU actually used by the backend (nvidia-smi), per-camera fps / model time
!python3 scripts/colab/colab_launch.py status

In [ ]:
# 9b. Live monitor: camera / analysed / live-view fps, latency, queue, drops, GPU util + memory, host RAM
#     one line per camera every 10 s, also written to /content/aivision-run/logs/monitor.csv.
#     If the backend stops, it prints the last log lines and the probable cause. Stop the cell to end.
!python3 scripts/colab/colab_launch.py monitor 10


In [ ]:
# 9c. Vehicle benchmark on YOUR footage (T4): input size x FP32/FP16 speed, detections, FP16 agreement.
#     1) capture 300 clean frames from the running Vehicle Detection session (camera id as shown in the UI)
#     2) STOP the detection in the UI (the benchmark needs the GPU alone), then run
#     3) apply the printed AIV_IMG_SIZE / AIV_HALF with: stop, then  %env AIV_IMG_SIZE=...  and start again
#     Optional --labels DIR (YOLO txt, COCO ids 1,2,3,5,7) gives real precision / recall and tunes conf + NMS.
CAM = "COLAB-CAM1"
!backend/.venv/bin/python scripts/colab/bench_vehicle.py capture --camera {CAM} --count 300 --interval 0.5


In [ ]:
# 9d. (detection stopped) run the benchmark on the captured frames
!backend/.venv/bin/python scripts/colab/bench_vehicle.py run --frames backend/data/bench/frames/{CAM} --imgsz 640,960,1280 --target-fps 15


In [ ]:
# Logs (backend, frontend, bridge, tailscaled, setup)
!tail -n 25 /content/aivision-run/logs/backend.log

In [ ]:
# 10. Stop everything cleanly (backend Ctrl+C semantics, bridge, VPN logout)
!python3 scripts/colab/colab_launch.py stop